In [1]:
# ============================================================
# MERGE CMEMS TEMPERATURE, SALINITY, AND AT NETCDF FILES
# EXPORT CLEANED CSV + SUMMARY LOG
# ============================================================

# -----------------------------
# 1. IMPORT LIBRARIES
# -----------------------------
import xarray as xr
import pandas as pd
import numpy as np

# -----------------------------

In [2]:

# 2. FILE PATHS
# -----------------------------
temp_file = "/Users/Mbongeleni/Library/CloudStorage/OneDrive-StellenboschUniversity/Documents/CSIR/Mercator/cmems_temperature.nc"

salt_file = "/Users/Mbongeleni/Library/CloudStorage/OneDrive-StellenboschUniversity/Documents/CSIR/Mercator/cmems_salinity.nc"

at_file = "/Users/Mbongeleni/Library/CloudStorage/OneDrive-StellenboschUniversity/Documents/CSIR/Mercator/AT.nc"

# ----------------------------

In [3]:

# 3. OPEN DATASETS
# -----------------------------
print("Opening datasets...")

ds_temp = xr.open_dataset(temp_file)
ds_salt = xr.open_dataset(salt_file)
ds_at = xr.open_dataset(at_file)

print("\nTemperature Dataset:")
print(ds_temp)

print("\nSalinity Dataset:")
print(ds_salt)

print("\nAT Dataset:")
print(ds_at)

# -----------------------------

Opening datasets...

Temperature Dataset:
<xarray.Dataset> Size: 267MB
Dimensions:    (time: 147, depth: 9, latitude: 421, longitude: 120)
Coordinates:
  * time       (time) datetime64[ns] 1kB 2009-12-01 2010-01-01 ... 2022-02-01
  * depth      (depth) float32 36B 0.494 1.541 2.646 3.819 ... 7.93 9.573 11.4
  * latitude   (latitude) float32 2kB -70.0 -69.92 -69.83 ... -35.08 -35.0
  * longitude  (longitude) float32 480B 15.0 15.08 15.17 ... 24.75 24.83 24.92
Data variables:
    thetao     (time, depth, latitude, longitude) float32 267MB ...
Attributes:
    Conventions:       CF-1.11
    title:             Monthly mean fields for product GLOBAL_REANALYSIS_PHY_...
    institution:       Mercator Ocean
    producer:          CMEMS - Global Monitoring and Forecasting Centre
    source:            MERCATOR GLORYS12V1
    credit:            E.U. Copernicus Marine Service Information (CMEMS)
    contact:           servicedesk.cmems@mercator-ocean.eu
    references:        http://marine.copern

In [4]:
# 4. KEEP ONLY REQUIRED VARIABLES
# -----------------------------
# thetao = temperature
# so = salinity
# AT = total alkalinity

temp = ds_temp[["thetao"]]
salt = ds_salt[["so"]]
at = ds_at[["AT"]]

# -----------------------------

In [5]:
# 5. MERGE DATASETS
# -----------------------------
print("\nMerging datasets...")

merged_ds = xr.merge([temp, salt, at])

print("\nMerged Dataset:")
print(merged_ds)

# -----------------------------


Merging datasets...

Merged Dataset:
<xarray.Dataset> Size: 802MB
Dimensions:    (time: 147, depth: 9, latitude: 421, longitude: 120)
Coordinates:
  * time       (time) datetime64[ns] 1kB 2009-12-01 2010-01-01 ... 2022-02-01
  * depth      (depth) float32 36B 0.494 1.541 2.646 3.819 ... 7.93 9.573 11.4
  * latitude   (latitude) float32 2kB -70.0 -69.92 -69.83 ... -35.08 -35.0
  * longitude  (longitude) float32 480B 15.0 15.08 15.17 ... 24.75 24.83 24.92
Data variables:
    thetao     (time, depth, latitude, longitude) float32 267MB ...
    so         (time, depth, latitude, longitude) float32 267MB ...
    AT         (time, depth, latitude, longitude) float32 267MB ...
Attributes:
    Conventions:       CF-1.11
    title:             Monthly mean fields for product GLOBAL_REANALYSIS_PHY_...
    institution:       Mercator Ocean
    producer:          CMEMS - Global Monitoring and Forecasting Centre
    source:            MERCATOR GLORYS12V1
    credit:            E.U. Copernicus Marin

In [6]:

# 6. CONVERT TO DATAFRAME
# -----------------------------
print("\nConverting to dataframe...")

df = merged_ds.to_dataframe().reset_index()

# -----------------------------


Converting to dataframe...


In [7]:
# 7. KEEP ONLY REQUIRED COLUMNS
# -----------------------------
required_columns = [
    "time",
    "depth",
    "latitude",
    "longitude",
    "so",
    "thetao",
    "AT"
]

df = df[required_columns]

# -----------------------------

In [8]:
# 8. SUMMARY BEFORE CLEANING
# -----------------------------
rows_before = len(df)

print("\n===================================")
print("SUMMARY BEFORE CLEANING")
print("===================================")
print(f"Total rows before cleaning: {rows_before:,}")

# -----------------------------
# 9. DROP NaN VALUES
# -----------------------------
print("\nDropping NaN values...")

df_clean = df.dropna()

# -----------------------------


SUMMARY BEFORE CLEANING
Total rows before cleaning: 66,837,960

Dropping NaN values...


In [9]:
# 10. SUMMARY AFTER CLEANING
# -----------------------------
rows_after = len(df_clean)

rows_dropped = rows_before - rows_after

print("\n===================================")
print("CLEANING SUMMARY")
print("===================================")
print(f"Rows before cleaning : {rows_before:,}")
print(f"Rows dropped (NaNs)  : {rows_dropped:,}")
print(f"Rows after cleaning  : {rows_after:,}")

# -----------------------------


CLEANING SUMMARY
Rows before cleaning : 66,837,960
Rows dropped (NaNs)  : 296,352
Rows after cleaning  : 66,541,608


In [10]:
# 11. SAVE CLEANED NC
# -----------------------------
output_nc = "/Users/Mbongeleni/Library/CloudStorage/OneDrive-StellenboschUniversity/Documents/CSIR/Mercator/merged_cleaned_data.nc"

# Convert cleaned dataframe back to xarray dataset
ds_clean = df_clean.set_index(
    ["time", "depth", "latitude", "longitude"]
).to_xarray()

# Save as NetCDF
ds_clean.to_netcdf(output_nc)


print("\n===================================")
print("FILE SAVED SUCCESSFULLY")
print("===================================")
print(f"\nNetCDF file saved successfully:\n{output_nc}")

# -----------------------------


FILE SAVED SUCCESSFULLY

NetCDF file saved successfully:
/Users/Mbongeleni/Library/CloudStorage/OneDrive-StellenboschUniversity/Documents/CSIR/Mercator/merged_cleaned_data.nc


In [11]:
# 12. OPTIONAL: SAVE SUMMARY LOG
# -----------------------------
summary_log = pd.DataFrame({
    "Metric": [
        "Rows before cleaning",
        "Rows dropped due to NaNs",
        "Rows after cleaning"
    ],
    "Value": [
        rows_before,
        rows_dropped,
        rows_after
    ]
})

log_file = "/Users/Mbongeleni/Library/CloudStorage/OneDrive-StellenboschUniversity/Documents/CSIR/Mercator/cleaning_summary_log.csv"

summary_log.to_csv(log_file, index=False)

print("\nSummary log saved to:")
print(log_file)

# -----------------------------


Summary log saved to:
/Users/Mbongeleni/Library/CloudStorage/OneDrive-StellenboschUniversity/Documents/CSIR/Mercator/cleaning_summary_log.csv


In [12]:
# 13. PREVIEW CLEAN DATA
# -----------------------------
print("\nPreview of cleaned dataframe:")
print(df_clean.head())


Preview of cleaned dataframe:
         time     depth  latitude  longitude         so    thetao           AT
73 2009-12-01  0.494025     -70.0  21.083345  32.880642 -1.858852  2258.379883
74 2009-12-01  0.494025     -70.0  21.166677  32.927944 -1.874233  2260.362549
75 2009-12-01  0.494025     -70.0  21.250011  32.937099 -1.919645  2260.928711
76 2009-12-01  0.494025     -70.0  21.333345  32.938629 -1.946745  2261.106445
77 2009-12-01  0.494025     -70.0  21.416677  32.918789 -1.958464  2260.349365
